# Structural neurotransmitter fingerprinting

In this tutorial, you will explore the structural neurotransmitter fingerprinting (SNTF) functionalities of Lacuna using the CLI.

**What you'll learn**:

- Fetch the neurotransmitter PET atlas and a structural connectome
- Prepare the atlas and precompute endpoint weights
- Compute NT-weighted structural disconnectivity scores
- Filter by neurotransmitter system

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/m-petersen/lacuna/blob/main/docs/tutorials/07-structural-neurotransmitter-fingerprinting.ipynb)

## Colab

Note: Colab provides limited computational resources. While these tutorials are designed to operate within those constraints, some Lacuna functionality cannot be fully demonstrated in this environment and requires access to higher-performance computing infrastructure.

Ignore this if you run this notebook locally.

In [ ]:
# --- Conda setup for Google Colab ---
# Lacuna's structural neurotransmitter fingerprinting relies on MRtrix3.
# However, Colab does not provide MRtrix3 out of the box.
# The condacolab package provides a workaround.
# This cell installs condacolab, which will restart the kernel.
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# Run this after the kernel restart. 
import sys

if 'google.colab' in sys.modules:

    import condacolab
    import subprocess
    condacolab.check()
    subprocess.run(["conda", "install", "-y", "-c", "mrtrix3", "mrtrix3"], check=True)
else:
    print("Not running in Colab — skipping condacolab setup.")

## Setup

In [ ]:
# Install Lacuna from GitHub
!pip install git+https://github.com/m-petersen/lacuna

# Install MRtrix3 via conda
!conda install -y -c mrtrix3 mrtrix3

Get the tutorial data.

In [ ]:
# Get tutorial data
!lacuna tutorial /tmp/tutorial_bids --force

Check whether MRtrix3 is properly installed as it is required for the analysis.

In [ ]:
!mrinfo --version

## Fetch data

SNTF requires two data sources:

1. **Neurotransmitter PET atlas** — curated representative PET receptor / transporter density maps from [NiSpace-data](https://github.com/LeonDLotter/NiSpace-data), pinned to a specific commit and verified by SHA-256.
2. **Structural connectome** — A normative tractogram of white matter fiber bundles (e.g., HCP1065).

*Note on the cerebellum:* in most PET maps the cerebellum (or a cerebellar grey-matter sub-region) is used as the kinetic-modelling reference — non-specific tracer binding there is divided out of every voxel, so the cerebellum's own values become uninterpretable and are masked out. Expect zero / NaN cerebellar values in the downloaded maps.

In [ ]:
# Fetch NT atlas
!lacuna fetch ntatlas \
    --output-dir /tmp/ntatlas_data

In [ ]:
# Fetch structural connectome
!lacuna fetch hcp1065 \
    --output-dir /tmp/hcp1065_data

## Precompute endpoint weights

`lacuna prepare sntf` precomputes per-streamline endpoint NT weights from the fetched atlas + tractogram. This step is slow because it samples the tractogram with MRtrix, but only runs once per (atlas, tractogram) pair.

In [ ]:
# Step 2: Precompute endpoint weights for the tractogram
!lacuna prepare sntf \
    --atlas-cache-dir /tmp/ntatlas_data \
    --connectome-path /tmp/hcp1065_data/hcp1065_1mm.tck \
    --cache-dir /tmp/sntf_cache

## Analysis

Structural neurotransmitter fingerprinting combines structural disconnection with neurotransmitter information. It:

1. Filters the normative tractogram by the lesion mask to identify disconnected streamlines
2. Looks up NT atlas values at the endpoints of those disconnected streamlines
3. Scores each neurotransmitter target based on the endpoint values

This answers: **what NT-weighted structural connectivity does the lesion disrupt?**

A high score for a given target indicates that the lesion disconnects pathways whose endpoints are rich in that neurotransmitter.

Run the analysis.

In [ ]:
!lacuna run sntf \
    /tmp/tutorial_bids/ \
    /tmp/outputs_sntf/ \
    --connectome-path /tmp/hcp1065_data/hcp1065_1mm.tck \
    --participant-label 01 \
    --mask-space MNI152NLin6Asym \
    --atlas-cache-dir /tmp/ntatlas_data

List the outputs.

In [ ]:
!ls /tmp/outputs_sntf/sub-01/ses-01/anat/

## Radar plot of the fingerprint

A radar plot is a natural way to visualise an NT fingerprint — each axis is a target, the radial extent is the score. We render one plot for sub-01 from the TSV produced above; the same code generalises to a loop over subjects.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

tsv = next(Path("/tmp/outputs_sntf/sub-01/ses-01/anat").glob("*sntf*profilestats.tsv"))
df = pd.read_csv(tsv, sep="\t")

targets = df["target"].tolist()
values = df["value"].to_numpy()
angles = np.linspace(0, 2 * np.pi, len(targets), endpoint=False)

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(projection="polar"))
ax.plot(np.append(angles, angles[0]), np.append(values, values[0]),
        color="steelblue", linewidth=1.5)
ax.fill(np.append(angles, angles[0]), np.append(values, values[0]),
        color="steelblue", alpha=0.25)
ax.set_xticks(angles)
ax.set_xticklabels(targets, fontsize=9)
ax.set_title("sub-01", pad=20)
ax.set_theta_zero_location("N")
ax.set_theta_direction(-1)
plt.tight_layout()
plt.show()
